## Notebook for the identification of the rsids for a given list of variants
1. Get list of variants based on chrom-pos-ref-alt
2. Identify the associated rsid 
3.  

In [1]:
import pandas as pd
import math
import yaml

# config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/global80K_config.yaml"
config_path = "../../global80K_config.yaml"
# load config file
with open(config_path, "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

# load helpful functions
import sys
sys.path.append('../../00_helpful_functions')
import helpful_functions as hf



In [4]:
from Bio import SeqIO
# from pyensembl import EnsemblRelease

# Function to check indexing of a variant
def check_variant_indexing(chrom, pos, ref, genome_fasta_path):
    """
    Check if a variant is 1-based or 0-based by comparing against the reference genome.

    Parameters:
    - chrom: Chromosome (e.g., "1")
    - pos: Position (integer, provided in the variant representation)
    - ref: Reference allele (string)
    - genome_fasta_path: Path to the reference genome in FASTA format.

    Returns:
    - "1-based", "0-based", or "unknown"
    """
    # Load the reference genome
    genome = SeqIO.to_dict(SeqIO.parse(genome_fasta_path, "fasta"))

    # Ensure chromosome naming matches the reference genome
    if chrom not in genome:
        raise ValueError(f"Chromosome {chrom} not found in reference genome.")

    # Extract the genome sequence for the chromosome
    sequence = genome[chrom].seq

    # Compare 1-based
    try:
        one_based_match = sequence[pos - 1 : pos - 1 + len(ref)] == ref
    except IndexError:
        one_based_match = False

    # Compare 0-based
    try:
        zero_based_match = sequence[pos : pos + len(ref)] == ref
    except IndexError:
        zero_based_match = False

    # Determine indexing
    if one_based_match and not zero_based_match:
        return "1-based"
    elif zero_based_match and not one_based_match:
        return "0-based"
    elif one_based_match and zero_based_match:
        return "Ambiguous (matches both)"
    else:
        return "Unknown (does not match either)"

In [ ]:
# variant_list_df['SPDI'].to_csv('spdi_significant_variants.csv', index=False, sep="\t", header=None)

In [5]:
def create_new_spdi(spdi):
    spdi_split = spdi.split(":")
    return f'{spdi_split[0]}:{int(spdi_split[1])+1}:{spdi_split[2]}:{spdi_split[3]}'

In [6]:
# read the variants:
variant_list_path = config['files']['collaborations']['langenberg_significant_vaiants_unique']
variant_list_df = pd.read_csv(variant_list_path, sep="\t")
variant_list_df

# variant_list_df['new_SPDI'] = variant_list_df['SPDI'].apply(create_new_spdi)
# variant_list_df

,SPDI,variant_logFC,variant_adj_P_Val
0,NC_000005.10:14408058:A:G,1.300521,1.617331e-132
1,NC_000005.10:14259915:C:T,1.119690,1.515352e-72
2,NC_000001.11:231734632:A:G,-0.948076,9.883252e-69
3,NC_000002.12:219267000:T:C,1.176764,5.617636e-64
4,NC_000003.12:142578094:C:G,0.723736,8.889886e-61
...,...,...,...
809,NC_000003.12:71497206:G:A,-0.128911,4.365701e-02
810,NC_000004.12:5758816:C:T,-0.128509,4.649902e-02
811,NC_000020.11:8366413:G:A,0.130123,4.878092e-02
812,NC_000010.11:113007099:T:A,0.133877,4.933347e-02


In [7]:
# get chrom and postion of table
def extract_chromosome_number(spdi):
    """Extract chromosome number or 'X' from SPDI identifier."""
    try:
        # Mapping of chromosome names to numbers
        chrom_map = {
            "NC_000001.11": "1", "NC_000002.12": "2", "NC_000003.12": "3", "NC_000004.12": "4",
            "NC_000005.10": "5", "NC_000006.12": "6", "NC_000007.14": "7", "NC_000008.11": "8",
            "NC_000009.12": "9", "NC_000010.11": "10", "NC_000011.10": "11", "NC_000012.12": "12",
            "NC_000013.11": "13", "NC_000014.9": "14", "NC_000015.10": "15", "NC_000016.10": "16",
            "NC_000017.11": "17", "NC_000018.10": "18", "NC_000019.10": "19", "NC_000020.11": "20",
            "NC_000021.9": "21", "NC_000022.11": "22", "NC_000023.11": "X", "NC_000024.10": "Y"
        }

        # Extract chromosome name from SPDI (before the first ':')
        chrom_name = spdi.split(':')[0]

        # Get corresponding chromosome number or "X" from the map
        chrom_number = chrom_map.get(chrom_name, chrom_name)  # Default to chrom_name if not found

        return chrom_number
    except Exception as e:
        return f"Error parsing chromosome from SPDI: {str(e)}"

def extract_one_based_position(spdi):
    try:
        # Split the SPDI by ':'
        parts = spdi.split(':')

        # Extract position
        position = int(parts[1]) + 1 # 1-based position for ensembl # Genomic position (e.g., 396320)

        return position
    except Exception as e:
        return f"Error parsing position from SPDI: {str(e)}"

def extract_ref(spdi):
    try:
        # Split the SPDI by ':'
        parts = spdi.split(':')

        # Extract reference
        reference = parts[2]  # Genomic reference (C)

        return reference
    except Exception as e:
        return f"Error parsing reference from SPDI: {str(e)}"

def extract_alt(spdi):
    try:
        # Split the SPDI by ':'
        parts = spdi.split(':')

        # Extract alternative
        alternative = parts[3]  # Genomic alternative (e.g., T)

        return alternative
    except Exception as e:
        return f"Error parsing alternative from SPDI: {str(e)}"

variant_list_df['chrom'] = variant_list_df['SPDI'].apply(extract_chromosome_number)
variant_list_df['pos_one_based'] = variant_list_df['SPDI'].apply(extract_one_based_position)
variant_list_df['ref'] = variant_list_df['SPDI'].apply(extract_ref)
variant_list_df['alt'] = variant_list_df['SPDI'].apply(extract_alt)

In [8]:
variant_list_df.loc[variant_list_df['pos_one_based'] == 396321]
# variant_list_df['pos_one_based'] = variant_list_df['pos'].apply(lambda pos: pos+1)
# variant_list_df

,SPDI,variant_logFC,variant_adj_P_Val,chrom,pos_one_based,ref,alt
42,NC_000006.12:396320:C:T,-0.540432,1.121085e-15,6,396321,C,T


In [11]:
# get rsid from position
# import myvariant

# # Initialize the MyVariant client
# mv = myvariant.MyVariant()

# # List of variants in chr-pos-ref-alt format
# variants = [
#     "6-396321-C-T",
#     "1-866422-C-T",
#     "1-876664-G-A",
#     "1-69635-G-C"
# ]

# # Convert variants to HGVS format
# hgvs_variants = [f"chr{var.split('-')[0]}:g.{var.split('-')[1]}{var.split('-')[2]}>{var.split('-')[3]}" for var in variants]

# # Fetch variant information
# results = mv.getvariants(hgvs_variants, fields="dbsnp.rsid")

# # Process and print results
# for variant, result in zip(variants, results):
#     rsid = result.get('dbsnp', {}).get('rsid', 'N/A')
#     print(f"Variant: {variant}, rsID: {rsid}")

# import requests

# def get_rsid_from_position(chromosome, position, species="human"):
#     """
#     Query Ensembl to retrieve the RSID for a given genomic position.

#     Args:
#         chromosome (str): Chromosome number (e.g., "1", "X").
#         position (int): Genomic position on the chromosome.
#         species (str): Species name (default: "human").

#     Returns:
#         str: RSID if found, or a message indicating no RSID.
#     """
#     # Construct the URL
#     url = f"https://rest.ensembl.org/overlap/region/{species}/{chromosome}:{position}-{position}?"

#     # Headers to specify JSON response
#     headers = {"Content-Type": "application/json"}

#     # Make the request
#     response = requests.get(url, headers=headers)

#     # Handle the response
#     if response.status_code == 200:
#         data = response.json()
#         # Look for RSID in the response
#         for item in data:
#             if item.get("id", "").startswith("rs"):
#                 return item["id"]
#         return "No RSID found for the given position."
#     else:
#         print(response)
#         return f"Error: Unable to fetch data (status code: {response.status_code})"

import requests

def get_rsid_from_position(chromosome, position, reference, alternative, species="homo_sapiens"):
    """
    Query the Ensembl REST API to retrieve the RSID for a given genomic position.

    Args:
        chromosome (str): Chromosome number (e.g., "1", "X").
        position (int): Genomic position on the chromosome.
        species (str): Species name (default: "homo_sapiens").

    Returns:
        str: RSID if found, or a message indicating no RSID.
    """
    # Construct the region string
    region = f"{chromosome}:{position}-{position}"

    # API endpoint URL
    url = f"https://rest.ensembl.org/overlap/region/{species}/{region}"

    # Query parameters
    params = {
        "feature": "variation"  # Limit to variation features (includes RSIDs)
    }

    # Headers to specify JSON response
    headers = {"Content-Type": "application/json"}

    try:
        # Make the request
        response = requests.get(url, headers=headers, params=params)

        # Raise an error for bad HTTP responses
        response.raise_for_status()

        # Parse the JSON response
        data = response.json()
        # Extract RSIDs based on given ref and alt alleles
        rsids = [item for item in data if item.get("id", "").startswith("rs") and reference in item.get('alleles', "") and alternative in item.get('alleles', "")]
        if len(rsids) > 1:
            print("Mutliple rsids for one SNV are not possible... Investigate them")
            print(chromosome, position, reference, alternative)
            for item in rsids:
                print(item.get("id", ""))
                print(item.get("alleles", ""))
        if rsids:
            return [rsid['id'] for rsid in rsids]
            return f"RSID(s) found: {', '.join(rsids)}"
        else:
            return None
            return "No RSID found for the given position."
    except requests.exceptions.RequestException as e:
        return f"Error querying Ensembl API: {e}"

variant_list_df['rsid'] = variant_list_df.apply(lambda row: get_rsid_from_position(row['chrom'], row['pos_one_based'], row['ref'], row['alt']), axis=1)
variant_list_df



# only on subset for testing:
# variant_list_df_head = variant_list_df.head(n=7)
# variant_list_df_head['rsid'] = variant_list_df_head.apply(lambda row: get_rsid_from_position(row['chrom'], row['pos'], row['ref'], row['alt']), axis=1)
# variant_list_df_head['n_rsid'] = variant_list_df_head['rsid'].apply(len)
# variant_list_df_head['chrom'].value_counts()
# variant_list_df_head
# variant_list_df_head = variant_list_df.head(n=100)
# variant_list_df_head['rsid'] = variant_list_df_head.apply(lambda row: get_rsid_from_position(row['chrom'], row['pos'], row['ref'], row['alt']), axis=1)
# variant_list_df_head
# variant_list_df_head.loc[variant_list_df_head['n_rsid'] >1]
# variant_list_df_head.loc[variant_list_df_head['n_rsid'] >1].apply(lambda row: get_rsid_from_position(row['chrom'], row['pos'], row['ref'], row['alt']), axis=1)
# # Example usage 6-396321-C-T
# chromosome = "6"
# position = 396321
# rsid = get_rsid_from_position(chromosome, position)
# print(f"RSID for {chromosome}:{position} is {rsid}")

# expected: rs2046495165 rs1740377627 (for the rows with multiple rsids)

Mutliple rsids for one SNV are not possible... Investigate them
4 107988194 G A
rs545917906
['G', 'A']
rs2108713616
['G', 'A']
Mutliple rsids for one SNV are not possible... Investigate them
4 107988148 A G
rs1223629127
['A', 'G']
rs2108713595
['A', 'G']
Mutliple rsids for one SNV are not possible... Investigate them
3 125214055 C T
rs1043159544
['C', 'A', 'T']
rs2125189856
['C', 'T']
Mutliple rsids for one SNV are not possible... Investigate them
3 125214056 C T
rs1444902715
['C', 'T']
rs2125189864
['C', 'T']


,SPDI,variant_logFC,variant_adj_P_Val,chrom,pos_one_based,ref,alt,rsid
0,NC_000005.10:14408058:A:G,1.300521,1.617331e-132,5,14408059,A,G,[rs1257445811]
1,NC_000005.10:14259915:C:T,1.119690,1.515352e-72,5,14259916,C,T,[rs113019087]
2,NC_000001.11:231734632:A:G,-0.948076,9.883252e-69,1,231734633,A,G,[rs2812395]
3,NC_000002.12:219267000:T:C,1.176764,5.617636e-64,2,219267001,T,C,[rs1951794311]
4,NC_000003.12:142578094:C:G,0.723736,8.889886e-61,3,142578095,C,G,[rs1027162076]
...,...,...,...,...,...,...,...,...
809,NC_000003.12:71497206:G:A,-0.128911,4.365701e-02,3,71497207,G,A,[rs62247035]
810,NC_000004.12:5758816:C:T,-0.128509,4.649902e-02,4,5758817,C,T,[rs12513194]
811,NC_000020.11:8366413:G:A,0.130123,4.878092e-02,20,8366414,G,A,[rs1249017274]
812,NC_000010.11:113007099:T:A,0.133877,4.933347e-02,10,113007100,T,A,[rs2045684499]


In [ ]:
### debugging
# chromosome = 6
# position = 396321
# species = 'homo_sapiens'
# region = f"{chromosome}:{position}-{position}"
# # Query parameters
# params = {
#     "feature": "variation"  # Limit to variation features (includes RSIDs)
# }

# # Headers to specify JSON response
# headers = {"Content-Type": "application/json"}
# # get_rsid_from_position(6, 396321, 'C', 'T')
# response = requests.get(f"https://rest.ensembl.org/overlap/region/{species}/{region}", headers=headers, params=params)
# print(response.json())
# get_rsid_from_position(4, 107988194, 'G', 'A')

[{'strand': 1, 'source': 'dbSNP', 'end': 396321, 'alleles': ['C', 'G', 'T'], 'assembly_name': 'GRCh38', 'feature_type': 'variation', 'id': 'rs12203592', 'seq_region_name': '6', 'start': 396321, 'consequence_type': 'intron_variant', 'clinical_significance': ['affects']}]


In [200]:
variant_list_df = variant_list_df[['SPDI', 'variant_logFC', 'variant_adj_P_Val', 'chrom', 'pos_one_based', 'ref', 'alt', 'rsid']]

In [201]:
variant_list_df.loc[variant_list_df['rsid'].isna()]

,SPDI,variant_logFC,variant_adj_P_Val,chrom,pos_one_based,ref,alt,rsid
94,NC_000004.12:119106333:G:A,0.316681,2.498489e-10,4,119106334,G,A,None


In [202]:
# only rsids:
variant_rsid_df = variant_list_df.loc[variant_list_df['rsid'].notna()]
print('Number of variants with an rsid: ', variant_rsid_df.shape[0]) # 680 813
variant_rsid_df['chrom'].value_counts()

Number of variants with an rsid:  813


chrom
1     87
6     65
3     53
14    51
7     49
11    47
16    44
2     39
15    36
10    34
9     31
18    30
5     30
17    29
8     28
12    27
22    26
4     26
X     20
19    20
13    16
20    14
21    11
Name: count, dtype: int64

In [203]:
# investigate cases with mutliple rsids for one variant
variant_rsid_df['n_rsid'] = variant_rsid_df['rsid'].apply(len)
variant_rsid_df.loc[variant_rsid_df['n_rsid'] > 1]

/tmp/ipykernel_185966/2845000223.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  variant_rsid_df['n_rsid'] = variant_rsid_df['rsid'].apply(len)


,SPDI,variant_logFC,variant_adj_P_Val,chrom,pos_one_based,ref,alt,rsid,n_rsid
99,NC_000004.12:107988193:G:A,0.299767,5.165735e-10,4,107988194,G,A,"[rs545917906, rs2108713616]",2
110,NC_000004.12:107988147:A:G,-0.317153,2.755070e-09,4,107988148,A,G,"[rs1223629127, rs2108713595]",2
114,NC_000003.12:125214054:C:T,0.359030,5.876085e-09,3,125214055,C,T,"[rs1043159544, rs2125189856]",2
187,NC_000003.12:125214055:C:T,0.369430,5.315528e-06,3,125214056,C,T,"[rs1444902715, rs2125189864]",2


In [ ]:
variant_rsid_df

,SPDI,variant_logFC,variant_adj_P_Val,chrom,pos,ref,alt,rsid,pos_one_based
0,NC_000005.10:14408058:A:G,1.300521,1.617331e-132,5,14408058,A,G,[rs1257445811],14408059
3,NC_000002.12:219267000:T:C,1.176764,5.617636e-64,2,219267000,T,C,[rs1951794311],219267001
4,NC_000003.12:142578094:C:G,0.723736,8.889886e-61,3,142578094,C,G,[rs1027162076],142578095
6,NC_000016.10:68784389:A:G,0.728294,3.003789e-47,16,68784389,A,G,[rs1176991852],68784390
8,NC_000022.11:42115773:T:C,-0.968261,9.525153e-42,22,42115773,T,C,[rs918000777],42115774
...,...,...,...,...,...,...,...,...,...
809,NC_000003.12:71497206:G:A,-0.128911,4.365701e-02,3,71497206,G,A,[rs62247035],71497207
810,NC_000004.12:5758816:C:T,-0.128509,4.649902e-02,4,5758816,C,T,[rs12513194],5758817
811,NC_000020.11:8366413:G:A,0.130123,4.878092e-02,20,8366413,G,A,[rs1249017274],8366414
812,NC_000010.11:113007099:T:A,0.133877,4.933347e-02,10,113007099,T,A,[rs2045684499],113007100


In [204]:
variant_rsid_df.loc[variant_rsid_df['pos_one_based'] == 396321]
# variant_rsid_df.loc[variant_rsid_df['chrom'] == '6']

,SPDI,variant_logFC,variant_adj_P_Val,chrom,pos_one_based,ref,alt,rsid,n_rsid
42,NC_000006.12:396320:C:T,-0.540432,1.121085e-15,6,396321,C,T,[rs12203592],1


In [ ]:
# variant_list_df[['SPDI', 'variant_logFC', 'variant_adj_P_Val', 'chrom', 'pos_one_based', 'ref', 'alt', 'rsid']].to_csv('20241119_significant_variants_with_rsid.tsv', sep="\t", index=False)

In [ ]:
import os
from Bio import Entrez
# from pydantic import BaseModel, Field
# from openai import OpenAI


# # Required: Export your API Key (on linux: export OPENAI_API_KEY="..." )
# client = OpenAI(
#   api_key=os.environ['OPENAI_API_KEY']
# )

# class Format(BaseModel):
#     rs_id: str = Field(..., description="rsid to check in the abstract")
#     rs_mentioned: bool = Field(..., description="True if the rsid is mentioned in the given abstract, otherwise false")
#     condition: str = Field(..., description="Associated medical condition or phenotype")
#     effect: str = Field(..., description="Effect of the rsid on the condition mentioned in the abstract")
#     celltype: str = Field(..., description="Most likely cell type the rsid is associated with")

# def check_abstract(rs_id_to_check, abstract):
#     prompt = f"""
#     You are provided with the following abstract:
#     ---
#     {abstract}
#     ---
#     Determine the following:
#     1. rs_id: "{rs_id_to_check}"
#     2. rs_id_mentioned: Does the given rsid "{rs_id_to_check}" appear in the abstract? Set it to True if found, otherwise False.
#     3. condition: The associated medical condition or phenotype mentioned in the abstract related to the rsid. Empty string if not mentioned.
#     4. effect: "The effect of the rsid "{rs_id_to_check}" on the condition mentioned in the abstract. Empty string if not mentioned"
#     5. celltype: "The cell type which the rsid and condition is associated with according to the abstract. Empty string if not mentioned."
#     """

#     # Query the model
#     response = client.beta.chat.completions.parse(
#         model="gpt-4o-mini",
#         messages=[
#             {"role": "system", "content": "You are a helpful assistant."},
#             {"role": "user", "content": prompt},
#         ],
#         response_format=Format,
#         #temperature=0,  # Deterministic output
#         #max_tokens=150,  # Adjust based on response size needed
#     )

#     output = response.choices[0].message.parsed
#     return output

def get_rsid_list(variant_list_df):
    """Concatenate all rsids within the lists of the rsid column"""
    return sum(variant_list_df["rsid"], [])

# Set your email (required by NCBI policy)
Entrez.email = "your_email@example.com"

# Function to fetch PubMed articles for an rsID
def fetch_pubmed_articles(rsid):
    query = f"{rsid}[All Fields]"
    handle = Entrez.esearch(db="pubmed", term=query, retmax=50)  # Adjust retmax as needed
    record = Entrez.read(handle)
    handle.close()
    # print(record)
    # Get list of PubMed IDs (PMIDs)
    pmids = record["IdList"]
    return pmids

# Function to fetch article abstract
def fetch_abstract(pmid):
    handle = Entrez.efetch(db="pubmed", id=pmid, rettype="abstract", retmode="text")
    abstract = handle.read()
    handle.close()
    return abstract

rsid_list = get_rsid_list(variant_list_df)

D = {}

for rsid in rsid_list:
    if rsid not in D.keys():
        D[rsid] = {}
    pmid_list = fetch_pubmed_articles(rsid)
    D[rsid] = pmid_list
    # for pmid in pmid_list:
    #     print(f"RSID: {rsid}, PubMed ID: {pmid}")
    #     abstract = fetch_abstract(pmid)
        # output = check_abstract(rsid, abstract)
        # print([output.condition, output.rs_mentioned, output.effect, output.celltype])
        # D[rsid][pmid] = [output.condition, output.rs_mentioned, output.effect, output.celltype]

print("OUTPUT:")
print(D)

OUTPUT:
{'rs1257445811': [], 'rs113019087': [], 'rs2812395': [], 'rs1951794311': [], 'rs1027162076': [], 'rs61882290': [], 'rs1176991852': [], 'rs3985938': [], 'rs918000777': [], 'rs1268570168': [], 'rs1349665746': [], 'rs2062724027': [], 'rs1479911567': [], 'rs1261941416': [], 'rs1649405845': [], 'rs10903341': [], 'rs929439316': [], 'rs1283017570': [], 'rs76488698': [], 'rs1323765603': [], 'rs938894630': [], 'rs1000609090': [], 'rs1810632615': [], 'rs2082523287': [], 'rs11833026': [], 'rs1342299671': [], 'rs556157384': [], 'rs1041896364': [], 'rs116137670': [], 'rs164858': [], 'rs888479076': [], 'rs11688390': [], 'rs17041224': [], 'rs4911546': [], 'rs904499030': [], 'rs963901877': [], 'rs76191879': [], 'rs2055256275': [], 'rs1355902955': [], 'rs80100717': [], 'rs76550327': [], 'rs1000841274': [], 'rs12203592': ['39075179', '37902747', '35390444', '34898573', '34424336', '34418235', '34293285', '33342058', '32856602', '32121219', '31958143', '31519034', '31246726', '30980179', '3052018

In [207]:
# check for elements in D which have more than length >= 1
# Count the number of RSIDs with list length >= 1
count_non_empty = sum(1 for value in D.values() if len(value) >= 1)

# Filter out RSIDs with empty lists
filtered_dict = {key: value for key, value in D.items() if len(value) >= 1}

print(f"Number of RSIDs with list length >= 1: {count_non_empty}")
print(f"Filtered dictionary: {filtered_dict}")

Number of RSIDs with list length >= 1: 3
Filtered dictionary: {'rs12203592': ['39075179', '37902747', '35390444', '34898573', '34424336', '34418235', '34293285', '33342058', '32856602', '32121219', '31958143', '31519034', '31246726', '30980179', '30520188', '29974532', '29753029', '29518100', '29315480', '29054604', '28755520', '28667740', '28502801', '28103633', '28083618', '27570521', '27435525', '26928068', '26857527', '25724930', '25705849', '25631878', '24906573', '24880832', '24631691', '23986280', '23948321', '23771755', '23548203', '23537197', '23393597', '22512251', '21962134', '21270109', '20602913', '19897031', '19396635', '18483556'], 'rs883125': ['35340006', '25760838', '19568425'], 'rs4765914': ['28792954', '26401721']}


In [232]:
def rsids_with_literature(rsid_list, target_rsids):
    """
    Checks if the target_rsids are within the rsid_list and returns False otherwise
    """
    for rsid in rsid_list:
        if rsid in target_rsids:
            return True
    return False

def rsids_without_literature(rsid_list, target_rsids):
    """
    Returns rsids without literature
    """
    for rsid in rsid_list:
        try:
            if rsid in target_rsids:
                return False
        except:
            for rsid_elem in rsid:
                if rsid in target_rsids:
                    return False

    return True


In [223]:
target_rsids = list(filtered_dict.keys())
target_rsids
variant_rsid_with_literature = variant_rsid_df.loc[variant_rsid_df["rsid"].apply(lambda rsids: rsids_with_literature(rsids, target_rsids))]
variant_rsid_with_literature

,SPDI,variant_logFC,variant_adj_P_Val,chrom,pos_one_based,ref,alt,rsid,n_rsid
42,NC_000006.12:396320:C:T,-0.540432,1.121085e-15,6,396321,C,T,[rs12203592],1
629,NC_000001.11:203177037:C:G,-0.209518,2.212148e-02,1,203177038,C,G,[rs883125],1
784,NC_000012.12:2311210:T:C,-0.193401,4.491360e-02,12,2311211,T,C,[rs4765914],1


In [225]:
variant_rsid_df.loc[variant_rsid_df["rsid"].apply(lambda rsids: rsids_without_literature(rsids, target_rsids))]

,SPDI,variant_logFC,variant_adj_P_Val,chrom,pos_one_based,ref,alt,rsid,n_rsid
0,NC_000005.10:14408058:A:G,1.300521,1.617331e-132,5,14408059,A,G,[rs1257445811],1
1,NC_000005.10:14259915:C:T,1.119690,1.515352e-72,5,14259916,C,T,[rs113019087],1
2,NC_000001.11:231734632:A:G,-0.948076,9.883252e-69,1,231734633,A,G,[rs2812395],1
3,NC_000002.12:219267000:T:C,1.176764,5.617636e-64,2,219267001,T,C,[rs1951794311],1
4,NC_000003.12:142578094:C:G,0.723736,8.889886e-61,3,142578095,C,G,[rs1027162076],1
...,...,...,...,...,...,...,...,...,...
809,NC_000003.12:71497206:G:A,-0.128911,4.365701e-02,3,71497207,G,A,[rs62247035],1
810,NC_000004.12:5758816:C:T,-0.128509,4.649902e-02,4,5758817,C,T,[rs12513194],1
811,NC_000020.11:8366413:G:A,0.130123,4.878092e-02,20,8366414,G,A,[rs1249017274],1
812,NC_000010.11:113007099:T:A,0.133877,4.933347e-02,10,113007100,T,A,[rs2045684499],1


In [221]:
len(rsids_without_literature)

814

In [ ]:
rsids_without_literature_df = variant_rsid_df.loc[variant_rsid_df["rsid"].apply(lambda rsids: rsids_without_literature(rsids, target_rsids))]
rsids_without_literature_df
rsids_without_literature = sum(rsids_without_literature_df["rsid"], [])
# len(rsids_without_literature) # 814
rsids_without_literature

TypeError: 'list' object is not callable

In [ ]:
# fetch_pubmed_articles('rs543065973')
# fetch_pubmed_articles('rs12203592')

{'Count': '48', 'RetMax': '48', 'RetStart': '0', 'IdList': ['39075179', '37902747', '35390444', '34898573', '34424336', '34418235', '34293285', '33342058', '32856602', '32121219', '31958143', '31519034', '31246726', '30980179', '30520188', '29974532', '29753029', '29518100', '29315480', '29054604', '28755520', '28667740', '28502801', '28103633', '28083618', '27570521', '27435525', '26928068', '26857527', '25724930', '25705849', '25631878', '24906573', '24880832', '24631691', '23986280', '23948321', '23771755', '23548203', '23537197', '23393597', '22512251', '21962134', '21270109', '20602913', '19897031', '19396635', '18483556'], 'TranslationSet': [], 'QueryTranslation': '"rs12203592"[All Fields]'}


['39075179', '37902747', '35390444', '34898573', '34424336', '34418235', '34293285', '33342058', '32856602', '32121219', '31958143', '31519034', '31246726', '30980179', '30520188', '29974532', '29753029', '29518100', '29315480', '29054604', '28755520', '28667740', '28502801', '28103633', '28083618', '27570521', '27435525', '26928068', '26857527', '25724930', '25705849', '25631878', '24906573', '24880832', '24631691', '23986280', '23948321', '23771755', '23548203', '23537197', '23393597', '22512251', '21962134', '21270109', '20602913', '19897031', '19396635', '18483556']

In [37]:
variant_list_df_head['rsid'].isna().sum()

58

#### Extend the number of rsids per variant with pseudo LD-blocks
- 

In [ ]:
all_common_dbsnp = '/home/kisa/coding/80K_MPRA/literature_for_rsids/00-common_all.vcf.gz'


In [12]:
variant_list_df.columns

Index(['SPDI', 'variant_logFC', 'variant_adj_P_Val', 'chrom', 'pos_one_based',
       'ref', 'alt', 'rsid'],
      dtype='object')

In [25]:
# add the right chromosome
def map_nc_to_chrom(nc_str):
    chrom = nc_str.split(".")[0].replace("NC_", "").lstrip("0")
    if chrom == "23":
        return "X"
    elif chrom == "24":
        return "Y"
    return chrom

def modified_SPDI(row, spdi_col, chrom_col):
    chrom, pos, ref, alt = row[spdi_col].split(':')
    return f"{row[chrom_col]}:{pos}:{ref}:{alt}"

variant_list_df['nc_chrom'] = variant_list_df['SPDI'].apply(lambda spdi: spdi.split(':')[0])
variant_list_df['n_chrom'] = variant_list_df['nc_chrom'].apply(map_nc_to_chrom)
# modify spdi because nc is not read by bedtools
variant_list_df['SPDI_mod'] = variant_list_df.apply(lambda row: modified_SPDI(row, spdi_col='SPDI', chrom_col='n_chrom'), axis=1)

In [ ]:
variant_list_df.head()

In [ ]:
import pandas as pd
import gzip

# Convert SPDI to BED
def spdi_to_bed(spdi_df, bed_output, window_size=500, var_column='SPDI'):
    bed_data = []
    for variant in spdi_df[var_column]:
        chrom, pos, ref, alt = variant.split(':')
        pos = int(pos)
        start = max(0, pos - window_size)  # 1kb window around the variant
        end = pos + window_size

        bed_data.append([chrom, start, end, variant])
    bed_df = pd.DataFrame(bed_data, columns=['chrom', 'start', 'end', 'variant'])
    bed_df.to_csv(bed_output, sep='\t', header=False, index=False)

# # Convert VCF to BED
# def vcf_to_bed(vcf_file_gz, bed_output):
#     with gzip.open(vcf_file_gz, 'rt') as vcf, open(bed_output, 'w') as bed:
#         for line in vcf:
#             if line.startswith("#"):
#                 continue
#             fields = line.strip().split('\t')
#             chrom, pos, id = fields[0], int(fields[1]), fields[2]
#             start = pos - 1  # VCF uses 1-based, BED uses 0-based
#             bed.write(f"{chrom}\t{start}\t{pos}\t{id}\n")

# # Find Overlaps
# def find_overlaps(significant_bed, vcf_bed, output_file):
#     import subprocess
#     cmd = f"bedtools intersect -a {significant_bed} -b {vcf_bed} -wa -wb > {output_file}"
#     subprocess.run(cmd, shell=True)
window_size = 1000
# File paths
vcf_file_gz = "/home/kisa/coding/80K_MPRA/literature_for_rsids/00-common_all.vcf.gz"
significant_bed = f"/home/kisa/coding/80K_MPRA/literature_for_rsids/significant_variants_w_{window_size}.bed"

# Execute
spdi_to_bed(variant_list_df, significant_bed, window_size=window_size, var_column='SPDI_mod')


# vcf_to_bed(vcf_file_gz, vcf_bed)
# find_overlaps(significant_bed, vcf_bed, output_file)


# bedtools intersect -a significant_variants_w_1000.bed -b variants.bed -wa -wb > overlapping_variants_w_1000.bed

# bedtools intersect -a significant_variants_w_10000.bed -b variants.bed -wa -wb > overlapping_variants_w_10000.bed

In [18]:
f"bedtools intersect -a {significant_bed} -b {vcf_bed} -wa -wb > {output_file}"

'bedtools intersect -a significant_variants.bed -b variants.bed -wa -wb > overlapping_variants.bed'

In [ ]:
variant_table_path = '/home/kisa/coding/80K_MPRA/literature_for_rsids/overlapping_variants_w_1000.bed'
# read variant file:
sig_variant_df = pd.read_csv(variant_table_path, sep="\t")
sig_variant_df.columns = ['chr_a', 'start_a', 'end_a', 'id_a', 'chr_b', 'start_b', 'end_b', 'id_b']

# this is required to merge all rsids using (rsid_list = sum(sig_variant_df['rsid'], []))
sig_variant_df['rsid'] = sig_variant_df['id_b'].apply(lambda elem: [elem])
# rsid_list = sum(sig_variant_df['rsid'], [])



In [34]:
sig_variant_df.head()

,chr_a,start_a,end_a,id_a,chr_b,start_b,end_b,id_b
0,5,14407058,14409058,5:14408058:A:G,5,14407225,14407226,rs147765965
1,5,14407058,14409058,5:14408058:A:G,5,14407246,14407247,rs544847272
2,5,14407058,14409058,5:14408058:A:G,5,14407368,14407369,rs147105342
3,5,14407058,14409058,5:14408058:A:G,5,14407539,14407540,rs73749238
4,5,14407058,14409058,5:14408058:A:G,5,14407798,14407799,rs545445630


In [ ]:
n_ld_rsids = sig_variant_df.id_b.nunique() # 18107
sig_variant_df.shape[0] # 21836
n_significant_rsids = sig_variant_df.id_a.nunique() # 808

n_significant_rsids

808

### Investigating the overlap

In [45]:
sig_variant_overlap = sig_variant_df[['id_a', 'id_b']].copy()
sig_variant_overlap_list = sig_variant_overlap.groupby('id_a').agg(list).reset_index().copy()

In [46]:
sig_variant_overlap_list

,id_a,id_b
0,10:10844519:C:T,"[rs141790331, rs185983960, rs183348286, rs9685..."
1,10:10891543:T:C,"[rs57684871, rs549412588, rs190265203, rs14771..."
2,10:110691243:A:T,"[rs144422020, rs148407381, rs560914088, rs1144..."
3,10:110764953:A:G,"[rs187460042, rs193045486, rs17762854, rs15063..."
4,10:110915759:G:A,"[rs553559194, rs79445680, rs536319529, rs49186..."
...,...,...
803,X:33027045:C:T,"[rs761732968, rs7884594, rs7884242, rs14216599..."
804,X:41265820:G:C,"[rs760526183, rs371012073, rs374330897, rs1811..."
805,X:53265913:G:A,"[rs782798042, rs138452722, rs782644615, rs1841..."
806,X:71152038:G:T,"[rs150091920, rs773104087, rs763066955, rs7363..."


,id_b,associated_id_as,num_overlaps
0,rs1000184,{1:8580771:G:C},1
1,rs10003676,{4:113023844:T:C},1
2,rs1000730,"{1:231828452:G:T, 1:231828442:C:T}",2
3,rs1000731,"{1:231828452:G:T, 1:231828442:C:T}",2
4,rs1000850,{9:93585575:G:A},1
...,...,...,...
18102,rs9980108,{21:37064205:C:A},1
18103,rs998091,{4:5686557:T:A},1
18104,rs9981091,{21:37452515:C:T},1
18105,rs9989879,{2:51080052:G:C},1


In [47]:
# Step 1: Flatten the `id_b` column for all significant rsIDs (id_a)
id_b_to_id_a_map = (
    sig_variant_overlap_list.explode("id_b")
    .groupby("id_b")["id_a"]
    .apply(set)  # Map each `id_b` to the set of `id_a` it overlaps with
    .reset_index(name="associated_id_as")
)

# Step 2: Count overlaps for each `id_b`
id_b_to_id_a_map["num_overlaps"] = id_b_to_id_a_map["associated_id_as"].apply(len)

# Step 3: Identify `id_b` with multiple `id_a` associations
id_b_with_multiple_overlaps = id_b_to_id_a_map[id_b_to_id_a_map["num_overlaps"] > 1]

# Step 4: Identify `id_a` with and without overlapping `id_b`
id_a_with_overlaps = (
    id_b_with_multiple_overlaps["associated_id_as"]
    .explode()
    .unique()
)
id_a_without_overlaps = set(sig_variant_overlap_list["id_a"]) - set(id_a_with_overlaps)

# Results
print("All id_b associations with overlaps:")
print(id_b_to_id_a_map)

print("\nid_b with multiple id_a overlaps:")
print(id_b_with_multiple_overlaps)

print("\nid_a with overlapping associated id_b:")
print(id_a_with_overlaps)

print("\nid_a without overlapping associated id_b:")
print(id_a_without_overlaps)

All id_b associations with overlaps:
             id_b                    associated_id_as  num_overlaps
0       rs1000184                     {1:8580771:G:C}             1
1      rs10003676                   {4:113023844:T:C}             1
2       rs1000730  {1:231828452:G:T, 1:231828442:C:T}             2
3       rs1000731  {1:231828452:G:T, 1:231828442:C:T}             2
4       rs1000850                    {9:93585575:G:A}             1
...           ...                                 ...           ...
18102   rs9980108                   {21:37064205:C:A}             1
18103    rs998091                     {4:5686557:T:A}             1
18104   rs9981091                   {21:37452515:C:T}             1
18105   rs9989879                    {2:51080052:G:C}             1
18106   rs9999575                   {4:108068603:A:G}             1

[18107 rows x 3 columns]

id_b with multiple id_a overlaps:
             id_b                                   associated_id_as  \
2       rs1000

In [52]:
id_b_to_id_a_map.sort_values(by='num_overlaps', ascending=False).head(100)

,id_b,associated_id_as,num_overlaps
4063,rs143083043,"{14:23437062:T:C, 14:23437063:G:C, 14:23437065...",6
13581,rs567642465,"{14:23437062:T:C, 14:23437063:G:C, 14:23437065...",6
13249,rs563502317,"{14:23437062:T:C, 14:23437063:G:C, 14:23437065...",6
13608,rs567889453,"{14:23437062:T:C, 14:23437063:G:C, 14:23437065...",6
16769,rs77151279,"{14:23437062:T:C, 14:23437063:G:C, 14:23437065...",6
...,...,...,...
14242,rs576085113,"{4:107988193:G:A, 4:107988058:T:G, 4:107988040...",4
16589,rs76489282,"{13:46866276:G:A, 13:46866333:C:T, 13:46866274...",4
14250,rs5761896,"{22:21014893:C:A, 22:21014939:C:T, 22:21014940...",4
17939,rs9442673,"{6:71474158:T:G, 6:71474101:G:T, 6:71474161:C:...",4


#### testing pickle 

In [ ]:
import pickle

pickle_path = '/home/kisa/coding/80K_MPRA/literature_for_rsids/rsid_to_pmid.pkl'
pickle_path = '/home/kisa/coding/80K_MPRA/literature_for_rsids/rsid_to_pmid_w_1000.pkl'

# Load the dictionary
with open(pickle_path, "rb") as f:
    D_loaded = pickle.load(f)

In [7]:
len(D_loaded.keys())

# check the length of the value lists
literature_count = 0
for key, value in D_loaded.items():
    if len(value) >= 1:
        literature_count += 1
        print(key, value)

print(f'Number of papers with literature: {literature_count}')

rs17602729 ['39125881', '38744035', '38053960', '37372415', '37253386', '36468031', '35921847', '35839336', '35337603', '35309536', '34356082', '33780152', '32569264', '32429460', '32379996', '30628539', '29520081', '29422864', '28572465', '26554440', '26529652', '26380113', '25682119', '24885427', '23681449', '21211004']
rs9936768 ['24768648']
rs7790964 ['37926287']
rs3738581 ['31876207']
rs59569785 ['24876173']
rs2735591 ['23784377']
rs577001 ['17374705']
rs216009 ['37489644']
rs11121179 ['18597038']
Number of papers with literature: 9
